# Test final YOLO26n imgsz=960 - Smart Traffic Agadir

Objectif: tester si `imgsz=960` ameliore la classe `person`, surtout les petits pietons, apres le modele final `800`.

Configuration:
- Dataset: `dataset_step3_tiny_person_crops.zip`
- Modele de depart prefere: `final_models/YOLO26n_step3_800_best.pt`
- Fallback: `runs/step3_tiny_person_yolo26n_realtime_800/weights/best.pt`
- Resolution: `imgsz=960`
- Epochs: `40`
- Sortie finale: `final_models/YOLO26n_step3_960_best.pt/.onnx`

On garde ce modele seulement si la classe `person` s'ameliore et si le FPS Raspberry Pi reste acceptable.

## 1. GPU et dependances

Dans Colab: `Runtime > Change runtime type > GPU`. A100/L4 ideal, T4 possible mais plus lent.

In [ ]:
!nvidia-smi
!python --version
!pip install -q ultralytics pyyaml onnx onnxslim

## 2. Monter Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Preparer dataset et chemins

Fichiers attendus dans Drive:

```text
/content/drive/MyDrive/thesis/dataset_step3_tiny_person_crops.zip
/content/drive/MyDrive/thesis/final_models/YOLO26n_step3_800_best.pt
```

Si le modele 800 n'est pas dans `final_models`, le notebook cherche automatiquement dans le dossier `runs`.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import shutil
import yaml

DATASET_NAME = 'dataset_step3_tiny_person_crops'
DRIVE_ROOT = Path('/content/drive/MyDrive/thesis')
DATA_ZIP = DRIVE_ROOT / f'{DATASET_NAME}.zip'

DATA_ROOT = Path('/content/datasets')
DATA_DIR = DATA_ROOT / DATASET_NAME
RUNS_DIR = DRIVE_ROOT / 'runs'
FINAL_DIR = DRIVE_ROOT / 'final_models'

MODEL_PATH_960 = FINAL_DIR / 'YOLO26n_step3_800_best.pt'
if not MODEL_PATH_960.exists():
    MODEL_PATH_960 = RUNS_DIR / 'step3_tiny_person_yolo26n_realtime_800' / 'weights' / 'best.pt'
if not MODEL_PATH_960.exists():
    MODEL_PATH_960 = DRIVE_ROOT / 'models' / 'YOLO26n_best.pt'

assert DATA_ZIP.exists(), f'Dataset introuvable: {DATA_ZIP}'
assert MODEL_PATH_960.exists(), f'Modele de depart introuvable: {MODEL_PATH_960}'

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

with ZipFile(DATA_ZIP, 'r') as zf:
    zf.extractall(DATA_ROOT)

assert DATA_DIR.exists(), f'Dossier dataset introuvable apres unzip: {DATA_DIR}'
assert (DATA_DIR / 'data.yaml').exists(), f'data.yaml introuvable: {DATA_DIR / "data.yaml"}'
assert (DATA_DIR / 'images/train').exists(), 'images/train introuvable'
assert (DATA_DIR / 'labels/train').exists(), 'labels/train introuvable'

print('Dataset pret:', DATA_DIR)
print('Modele de depart:', MODEL_PATH_960)

## 4. Corriger `data.yaml` pour Colab

In [ ]:
DATA_YAML = DATA_DIR / 'data_colab.yaml'

with open(DATA_DIR / 'data.yaml', 'r') as f:
    data = yaml.safe_load(f)

data['path'] = str(DATA_DIR)
data['train'] = 'images/train'
data['val'] = 'images/val'
data['test'] = 'images/test'

with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(DATA_YAML.read_text())

## 5. Audit rapide des splits

In [ ]:
from collections import Counter

names = data['names']

def count_split(split):
    counts = Counter()
    labels = sorted((DATA_DIR / 'labels' / split).glob('*.txt'))
    images = list((DATA_DIR / 'images' / split).glob('*'))
    for label in labels:
        for line in label.read_text().splitlines():
            parts = line.split()
            if parts:
                counts[names[int(float(parts[0]))]] += 1
    return len(images), len(labels), counts

for split in ['train', 'val', 'test']:
    image_count, label_count, counts = count_split(split)
    print(split, 'images=', image_count, 'labels=', label_count, dict(counts))

## 6. Fine-tuning YOLO26n en 960

Si Colab affiche une erreur memoire sur T4, remplace `batch=-1` par `batch=8`.

In [ ]:
from ultralytics import YOLO

RUN_NAME = 'step3_tiny_person_yolo26n_realtime_960'
IMGSZ = 960
EPOCHS = 40

model = YOLO(str(MODEL_PATH_960))

train_kwargs = dict(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=-1,
    patience=10,
    workers=2,
    device=0,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    optimizer='AdamW',
    lr0=0.0001,
    lrf=0.1,
    mosaic=0.1,
    close_mosaic=5,
    cos_lr=True,
    seed=42,
    deterministic=False,
    cache=False,
)

print('Run test 960:', RUN_NAME)
print('Modele de depart:', MODEL_PATH_960)
print('Arguments:', train_kwargs)
results = model.train(**train_kwargs)

## 7. Validation officielle val/test

In [ ]:
best_pt = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
assert best_pt.exists(), f'best.pt introuvable: {best_pt}'

best_model = YOLO(str(best_pt))

val_metrics = best_model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=IMGSZ,
    device=0,
    plots=True,
    project=str(RUNS_DIR),
    name=f'{RUN_NAME}_val',
    exist_ok=True,
)

test_metrics = best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=IMGSZ,
    device=0,
    plots=True,
    project=str(RUNS_DIR),
    name=f'{RUN_NAME}_test',
    exist_ok=True,
)

print('Best model:', best_pt)
print('Val mAP50-95:', val_metrics.box.map)
print('Val mAP50:', val_metrics.box.map50)
print('Test mAP50-95:', test_metrics.box.map)
print('Test mAP50:', test_metrics.box.map50)
print()
print('Compare avec 800: Test mAP50=0.775, Test mAP50-95=0.563, person recall=0.291, person mAP50=0.450')

## 8. Export ONNX et copie finale

In [ ]:
onnx_path = Path(best_model.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True))
assert onnx_path.exists(), f'Export ONNX introuvable: {onnx_path}'

final_pt = FINAL_DIR / 'YOLO26n_step3_960_best.pt'
final_onnx = FINAL_DIR / 'YOLO26n_step3_960_best.onnx'
final_results = FINAL_DIR / 'YOLO26n_step3_960_results.csv'

shutil.copy2(best_pt, final_pt)
shutil.copy2(onnx_path, final_onnx)

results_csv = RUNS_DIR / RUN_NAME / 'results.csv'
if results_csv.exists():
    shutil.copy2(results_csv, final_results)

print('PT final:', final_pt)
print('ONNX final:', final_onnx)
if final_results.exists():
    print('CSV final:', final_results)

## 9. Fichiers a recuperer

Apres entrainement, telecharger depuis Drive:

```text
/content/drive/MyDrive/thesis/final_models/YOLO26n_step3_960_best.pt
/content/drive/MyDrive/thesis/final_models/YOLO26n_step3_960_best.onnx
/content/drive/MyDrive/thesis/final_models/YOLO26n_step3_960_results.csv
```

Test pipeline avec ce modele:

```bash
python3 pipeline/main.py \
  --source data/videos/ne8th/Bellevue_Bellevue_NE8th__2017-09-11_14-08-31_3min.mp4 \
  --model models/downloads/YOLO26n_step3_960_best.onnx \
  --imgsz 960
```

Decision finale:
- Si `person` augmente et FPS Pi acceptable: garder 960.
- Si 960 est trop lent: garder 800 comme modele embarque, et utiliser 960 dans le benchmark accuracy.